In [2]:
import pandas as pd
from google import genai # <-- Menggunakan library baru
import json
from transformers import pipeline
import os
from dotenv import load_dotenv

load_dotenv()

api=os.getenv("API")
# ==========================================
# 1. INISIALISASI KEDUA MODEL
# ==========================================
# Inisialisasi client Gemini versi terbaru
# Ganti dengan API Key yang kamu dapatkan dari Google AI Studio
client = genai.Client(api_key=api)

# Load model DistilBERT andalanmu
sentiment_analyzer = pipeline(
    "text-classification", 
    model="./model_new/checkpoint-10000", 
    tokenizer="distilbert-base-uncased"
)

def get_hybrid_prediction(text):
    # ==========================================
    # 2. FASE 1: EKSTRAKSI OLEH LLM (Gemini Flash)
    # ==========================================
    prompt = f"""
    Kamu adalah asisten AI ahli ekstraksi informasi tata bahasa.
    Tugasmu HANYA mengekstrak Aspek (segala hal yang dikomentari pengguna, seperti makanan, minuman, pelayanan, suasana restoran, harga, dll) dan Opini (penilaian) dari ulasan pengguna.
    
    Aturan ketat:
    1. Ekstrak SEGALA aspek yang memiliki opini, jangan batasi hanya pada makanan.
    2. Pisahkan aspek yang dirangkai dengan kata hubung (contoh: "suasana dan pelayanan").
    3. Tangkap kata penegas/negasi secara lengkap pada opini (contoh: "too sweet", "not good", "very cozy", "don't like").
    4. Jika opini berupa kata kerja (contoh: "love", "hate"), anggap itu sebagai opini.
    5. Kembalikan HANYA dalam format JSON array seperti contoh. TIDAK BOLEH menebak sentimen.
    
    Contoh Output:
    [
        {{"aspek": "cake", "opini": "too sweet"}},
        {{"aspek": "restaurant vibe", "opini": "very cozy"}},
        {{"aspek": "cashier", "opini": "not polite"}}
    ]

    Teks Ulasan: "{text}"
    """
    
    try:
        # Cara memanggil model menggunakan syntax library yang baru
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt
        )
        
        # Membersihkan output JSON dari LLM
        clean_json = response.text.replace('```json', '').replace('```', '').strip()
        data_ekstraksi = json.loads(clean_json)
        
        hasil_akhir = []
        
        # ==========================================
        # 3. FASE 2: KLASIFIKASI OLEH DISTILBERT
        # ==========================================
        for item in data_ekstraksi:
            aspect = item.get("aspek", "")
            opinion = item.get("opini", "")
            
            if aspect and opinion:
                kalimat_fokus = f"The {aspect} is {opinion}"
                
                prediksi = sentiment_analyzer(kalimat_fokus)[0]
                label = "Positif" if prediksi['label'] == 'LABEL_1' else "Negatif"
                skor = f"{prediksi['score'] * 100:.1f}%"
                
                hasil_akhir.append({
                    "Aspek (Target)": aspect.capitalize(),
                    "Opini Pengguna": opinion.lower(),
                    "Sentimen (DistilBERT)": label, 
                    "Keyakinan Model": skor
                })
                
        if not hasil_akhir:
             return pd.DataFrame([{"Pesan": "Gagal mengekstrak aspek dan opini."}])
             
        return pd.DataFrame(hasil_akhir)
        
    except Exception as e:
        return pd.DataFrame([{"Pesan": f"Terjadi kesalahan sistem: {str(e)}"}])

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [3]:
# ==========================================
# UJI COBA ARSITEKTUR HYBRID FINAL
# ==========================================
ulasan_user = "Overall, Emilia Cucina Italiana is a wonderful place not only for great food, but also for outstanding hospitality and special moments. I would highly recommend it and would be happy to return."
df_hasil = get_hybrid_prediction(ulasan_user)
display(df_hasil)

,Aspek (Target),Opini Pengguna,Sentimen (DistilBERT),Keyakinan Model
0,Place,wonderful,Positif,99.8%
1,Food,great,Positif,99.8%
2,Hospitality,outstanding,Positif,99.7%
3,Emilia cucina italiana,highly recommend,Positif,99.7%
4,Emilia cucina italiana,happy to return,Positif,99.3%
